# Held-Out Selection Request — ML-1M / LightGCN

Camera-ready requirement #2 (Reviewer 1; also Reviewer 5 point 4): checkpoint
selection by MI-NG on the **actual** deletion request leaks the evaluation
target into model selection. This notebook reruns AIE and GAIE with a
**leakage-free protocol** and reports how much the numbers move.

### Protocol
1. **Selection request** = a *fixed simulated* deletion set, drawn once with a
   dedicated seed (`--selection_seed`, independent of the training seed),
   **size-matched** to the actual request (same deletion rate), and with the
   injected adversarial edges **excluded from the draw** — so the selection
   request shares no edge with the evaluation target.
2. **Training simulations** also exclude the injected edges
   (`--exclude_adv_from_sim`), closing the reviewer's second leakage path
   ("simulated sets are sampled from E, which contains the injected edges").
3. Every checkpoint decision — in **both** the pre-training and the
   fine-tuning stage — uses MI-NG on the held-out selection request.
4. The **actual request is evaluated exactly once**, at the end, on the best
   checkpoint. That number is the honest headline result.

Requires the repo at a commit that includes `--held_out_selection` /
`--selection_seed` / `--exclude_adv_from_sim` in `config/params.py` and the
matching coach + DataHandler support.

Scope: ML-1M / LightGCN, seed 1234 (matching the main tables). ~2 h on a
Kaggle T4. Results → `./logs/heldout_selection_ml1m_lightgcn.json`.
If `./logs/eff_var_ml1m_lightgcn.json` (efficiency/variance notebook) is
present, a final cell prints the actual-request-selection vs held-out-selection
comparison table for the paper.

### Hyperparameters
Inline in Section 1 (same values as the efficiency/variance notebook) — no
datasets to attach, no secrets to configure. Identical values by design: the
selection signal must be the only variable between the two protocols.

### Kaggle setup
Settings → Accelerator → **GPU**. That's all.

---
## 0. Environment

In [ ]:
import os
import subprocess
import torch
cuda_tag = torch.version.cuda.replace(".", "")        # e.g. "121" or "124"
torch_tag = ".".join(torch.__version__.split(".")[:2]) # e.g. "2.6"
whl_url = f"https://data.pyg.org/whl/torch-{torch_tag}.0+cu{cuda_tag}.html"
print(f"Installing torch-scatter + torch-sparse from: {whl_url}")
subprocess.check_call(["pip", "install", "-q", "torch-scatter", "torch-sparse", "-f", whl_url])
subprocess.check_call(["pip", "install", "-q", "setproctitle"])

import torch_scatter
import torch_sparse
import setproctitle
print("torch_scatter version:", torch_scatter.__version__)
print("torch_sparse version:", torch_sparse.__version__)
print("setproctitle installed \u2713")

In [ ]:
import os

REPO_URL = "https://github.com/Shuvayu12/unlearnrec_improv.git"
PROJECT_DIR = "/kaggle/working/unlearnrec_improv"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repo already cloned.")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
import sys

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Clear sys.argv so argparse in config/params.py doesn't choke on notebook kernel args
sys.argv = [sys.argv[0]]

from config.params import args
from data.data_handler import DataHandler
from Utils.time_logger import log
from models.Model import LightGCN, AIE, GAIE
from training.pretrain_simgcl import Coach as PretrainCoach   # dispatches on args.model
from unlearning.aie_unlearn import Coach as AIECoach
from unlearning.gaie_unlearn import Coach as GAIECoach

import torch as t
import numpy as np
import random
import time
import json
import glob
import traceback

# Fail fast if the repo commit lacks the held-out selection support
assert hasattr(args, 'held_out_selection'), (
    'This repo clone has no --held_out_selection support; pull the commit '
    'that adds it to config/params.py, data/data_handler.py and the coaches.')

os.makedirs("./ckpt", exist_ok=True)
os.makedirs("./logs", exist_ok=True)

print(f"CUDA available: {t.cuda.is_available()}")
if t.cuda.is_available():
    print(f"GPU: {t.cuda.get_device_name(0)}")

---
## 1. Hyperparameters (inline — zero setup)

Identical to the efficiency/variance notebook: the effective ml1m/lightgcn
configuration of the committed 3x3 notebooks (`UNLEARN_CFG` + the
`STRONG_UNLEARN` override for LightGCN combos). Keeping the values byte-for-byte
identical is deliberate — the selection signal is the only variable this
experiment changes.

Key ↔ paper-symbol map: `epoch`→N_pre, `pretrain_drop_rate`→ρ_s,
`sim_epoch`→T_sim, `unlearn_wei`→λ_u, `align_wei`→λ_p, `align_temp`→τ,
`rec_wei`→λ_r (GAIE), fine-tune `epoch`→N_ft, `unlearn_wei`→λ'_u,
`align_wei`→λ'_p, `reg`→γ.

**Optional override (not required):** an `hparams.json` via `HPARAMS_PATH` or
an attached Kaggle input dataset replaces any non-null keys. With nothing
attached, the defaults just run.

In [ ]:
# Effective ml1m/lightgcn configuration, identical to the committed 3x3
# notebooks (UNLEARN_CFG + STRONG_UNLEARN override for LightGCN combos)
# and to the efficiency/variance notebook.
HP = {
    'unlearn':       dict(epoch=30, batch=4096, pretrain_drop_rate=0.1,
                          sim_epoch=10, unlearn_wei=1.0, align_wei=0.01,
                          align_temp=1.0, perf_degrade=0.3),
    'gaie_extra':    dict(rec_wei=0.03, perf_degrade=0.5),
    'finetune_aie':  dict(epoch=20, unlearn_wei=0.2, align_wei=0.01),
    'finetune_gaie': dict(epoch=15, unlearn_wei=0.2, align_wei=0.01),
    'backbone':      dict(pretrain_epoch=50, batch=2048, reg=1e-7,
                          adv_method='lightgcn0.5'),
}
HP_SOURCE = 'inline notebook defaults (ml1m/lightgcn, matches the 3x3 notebooks)'

# Optional override -- no setup required. If an hparams.json exists at
# HPARAMS_PATH or inside an attached Kaggle input dataset, its non-null
# values replace the defaults.
_cand = os.environ.get('HPARAMS_PATH')
if not (_cand and os.path.exists(_cand)):
    _hits = sorted(glob.glob('/kaggle/input/*/hparams.json'))
    _cand = _hits[0] if _hits else None
if _cand:
    with open(_cand) as fs:
        _ext = json.load(fs)
    for _sec, _vals in _ext.items():
        HP.setdefault(_sec, {}).update(
            {k: v for k, v in _vals.items() if v is not None})
    HP_SOURCE = f'inline defaults overridden by {_cand}'

print('Hyperparameters:', HP_SOURCE)

---
## 2. Configuration & harness

In [ ]:
DATASET = 'ml1m'
BACKBONE = 'lightgcn'
SEED = 1234                # match the main tables
SELECTION_SEED = 5678      # independent draw for the held-out request
PIPELINES = ['aie', 'gaie']
FT_TUNE_E0 = True

RESULTS_FILE = './logs/heldout_selection_ml1m_lightgcn.json'
all_results = {}
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as fs:
        all_results = json.load(fs).get('results', {})
    print(f'Resuming: {len(all_results)} entries already in {RESULTS_FILE}')


def save_results():
    with open(RESULTS_FILE, 'w') as fs:
        json.dump({'dataset': DATASET, 'backbone': BACKBONE, 'seed': SEED,
                   'selection_seed': SELECTION_SEED, 'ft_tune_e0': FT_TUNE_E0,
                   'protocol': 'held-out simulated selection request; '
                               'adversarial edges excluded from simulations; '
                               'actual request evaluated once at the end',
                   'results': all_results}, fs, indent=2)


def reset_seeds(seed):
    t.manual_seed(seed)
    t.cuda.manual_seed_all(seed)
    t.backends.cudnn.deterministic = True
    np.random.seed(seed)
    random.seed(seed)


def apply_args(d):
    for k, v in d.items():
        setattr(args, k, v)


def set_common(seed=SEED):
    args.data = DATASET
    args.model = BACKBONE
    args.seed = seed
    args.gpu = '0'
    args.lr = 1e-3
    args.latdim = 128
    args.gnn_layer = 3
    args.topk = 20
    args.tst_bat = 256
    args.decay = 1.0
    args.bpr_wei = 1.0
    args.load_model = None
    args.reg = HP['backbone']['reg']
    args.adv_method = HP['backbone']['adv_method']
    args.adversarial_attack = True
    # --- the leakage-free protocol ---
    args.held_out_selection = True
    args.selection_seed = SELECTION_SEED
    args.exclude_adv_from_sim = True


UNLEARN_STRUCTURAL = dict(
    lr=1e-3, test_drop_rate=0.003, tst_epoch=5, bpr_wei=1.0,
    align_type='v2', unlearn_type='v1', overall_withdraw_rate=0.1,
    withdraw_rate_init=1, hyper_temp=1.0, unlearn_ssl=1e-3,
    layer_mlp=2, leaky=0.99, act='leaky', unlearn_layer=0,
)


def unlearn_cfg(pipeline):
    cfg = dict(UNLEARN_STRUCTURAL)
    u = HP['unlearn']
    cfg.update(epoch=u['epoch'], batch=u['batch'],
               pretrain_drop_rate=u['pretrain_drop_rate'], sim_epoch=u['sim_epoch'],
               unlearn_wei=u['unlearn_wei'], align_wei=u['align_wei'],
               align_temp=u['align_temp'], perf_degrade=u['perf_degrade'])
    if pipeline == 'gaie':
        g = HP['gaie_extra']
        cfg['rec_wei'] = g['rec_wei']
        cfg['perf_degrade'] = g.get('perf_degrade', cfg['perf_degrade'])
    return cfg


print(f'Config: {DATASET}/{BACKBONE}, seed={SEED}, selection_seed={SELECTION_SEED}, '
      f'pipelines={PIPELINES}')

---
## 3. Attacked backbone (shared with the other notebooks)

In [ ]:
set_common()
args.epoch = HP['backbone']['pretrain_epoch']
args.batch = HP['backbone']['batch']
args.tst_epoch = 10
BACKBONE_CKPT = f'./ckpt/pre_{DATASET}_{BACKBONE}_adv'
args.save_path = BACKBONE_CKPT
reset_seeds(SEED)

handler_atk = DataHandler()
handler_atk.load_data(drop_rate=0.0, adv_attack=True)
print(f'users={args.user}, items={args.item}, '
      f'injected adversarial edges={len(handler_atk.adv_edges[0])}')

if not os.path.exists(BACKBONE_CKPT + '.mod'):
    _t0 = time.time()
    PretrainCoach(handler_atk).run()
    print(f'Attacked backbone trained in {time.time() - _t0:.1f}s')
else:
    print(f'Reusing checkpoint {BACKBONE_CKPT}.mod')

_m = t.load(BACKBONE_CKPT + '.mod', weights_only=False)['model'].cuda()
_m.training = False
BACKBONE_RES = dict(PretrainCoach(handler_atk).tst_epoch(_m))
print(f'[Attacked backbone] Recall@{args.topk}={BACKBONE_RES["Recall"]:.4f}, '
      f'NDCG@{args.topk}={BACKBONE_RES["NDCG"]:.4f}')
del _m, handler_atk
t.cuda.empty_cache()

---
## 4. Leakage-free pipelines

Both stages run with `held_out_selection=True`: every `sim_epoch` the coach
evaluates MI-NG on the fixed held-out simulated request (the `[selection]`
lines in the log) and checkpoints on that. The actual request is evaluated
**once**, below, on the best checkpoint — look for the
`final (actual request, first look)` block in each pipeline's output.

In [ ]:
COACHES = {'aie': AIECoach, 'gaie': GAIECoach}


def run_heldout(pipeline):
    coach_cls = COACHES[pipeline]

    # ---- stage 1: encoder pre-training (held-out selection) ----
    set_common()
    apply_args(unlearn_cfg(pipeline))
    args.fineTune = False
    args.trained_model = BACKBONE_CKPT
    args.save_path = f'./ckpt/{pipeline}_{DATASET}_{BACKBONE}_heldout'
    reset_seeds(SEED)
    h_u = DataHandler()
    # Simulated initial drop (NOT the actual request): any training epoch that
    # runs before the first sim-epoch reload must never see the request's
    # edges as its deletion set. adv_edges are still loaded (the trn file is
    # the attacked matrix), so selection-request sizing keeps working.
    h_u.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=False, exclude_adv=True)
    coach = coach_cls(h_u)
    _t0 = time.time()
    coach.run()
    t_unlearn = time.time() - _t0

    # ---- stage 2: fine-tuning (E0 unfrozen, held-out selection) ----
    args.fineTune = True
    args.ft_tune_e0 = FT_TUNE_E0
    args.model_2_finetune = args.save_path
    ft = HP[f'finetune_{pipeline}']
    apply_args(dict(epoch=ft['epoch'], unlearn_wei=ft['unlearn_wei'],
                    align_wei=ft['align_wei']))
    args.save_path = args.save_path + '_ft'
    reset_seeds(SEED)
    h_f = DataHandler()
    # Critical: the fine-tune coach starts its epoch loop at a negative stloc
    # (-(tst_epoch-1)), so a few trn_epoch() calls run BEFORE the first
    # sim-epoch reload. Under the old protocol (adv_attack=True here) those
    # epochs optimize L_u directly on the actual request's edges. Loading a
    # simulated drop closes that leak.
    h_f.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=False, exclude_adv=True)
    coach = coach_cls(h_f)
    _t0 = time.time()
    coach.run()
    t_ft = time.time() - _t0

    # ---- the ONE evaluation on the actual request ----
    h_e = DataHandler()
    h_e.load_data(drop_rate=args.test_drop_rate, adv_attack=True)
    sp = args.save_path + '.mod'
    if os.path.exists(sp):
        coach.model = t.load(sp, weights_only=False)['model'].cuda()
    coach.handler = h_e
    coach.model.eval()      # GAIE: Z = mu at inference (paper Sec 3.2)
    reset_seeds(SEED)       # reproducible evaluation negatives
    metrics = coach.tst_epoch(coach.model)
    mi = coach.test_unlearn(
        coach.model, prefix=f'[{pipeline.upper()}] final (actual request, first look)')

    row = dict(selection='held-out simulated',
               Recall=metrics['Recall'], NDCG=metrics['NDCG'],
               MI_BF=mi['mi_bf'], MI_NG=mi['mi_ng'],
               MI_AUC=mi['mi_auc'], MI_ACC=mi['mi_acc'],
               BeforeProb=mi['avg_before_prob'], AfterProb=mi['avg_after_prob'],
               NegProb=mi['avg_neg_prob'],
               UnlearnTime=t_unlearn, FinetuneTime=t_ft,
               BackboneRecall=BACKBONE_RES['Recall'], BackboneNDCG=BACKBONE_RES['NDCG'])
    del coach, h_u, h_f, h_e
    t.cuda.empty_cache()
    return row


for pipeline in PIPELINES:
    key = f'{pipeline}/heldout'
    if key in all_results and 'error' not in all_results[key]:
        print(f'skip {key} (already done)')
        continue
    print('\n' + '#' * 100)
    print(f'### {pipeline.upper()} — held-out selection — {DATASET}/{BACKBONE} seed {SEED}')
    print('#' * 100)
    try:
        all_results[key] = run_heldout(pipeline)
    except Exception:
        traceback.print_exc()
        all_results[key] = {'error': traceback.format_exc()[-1500:]}
    save_results()
    print(f'saved {RESULTS_FILE}')

---
## 5. Comparison: actual-request selection vs held-out selection

The table the paper needs. The actual-request-selection baseline is pulled from
the efficiency/variance notebook's JSON (same dataset/backbone/seed/
hyperparameters — the selection signal is the only difference). If that file
is absent, the held-out rows are printed alone; fill the baseline column from
the main-table run.

In [ ]:
METRICS = ['Recall', 'NDCG', 'MI_BF', 'MI_NG', 'MI_AUC', 'MI_ACC']

baseline = {}
BASELINE_FILE = './logs/eff_var_ml1m_lightgcn.json'
if os.path.exists(BASELINE_FILE):
    with open(BASELINE_FILE) as fs:
        _eff = json.load(fs).get('results', {})
    for p in PIPELINES:
        r = _eff.get(f'{p}/seed{SEED}')
        if r and 'error' not in r:
            baseline[p] = r
    print(f'Baseline (actual-request selection) loaded from {BASELINE_FILE}: '
          f'{list(baseline)}')
else:
    print(f'{BASELINE_FILE} not found — run the efficiency/variance notebook '
          f'for the baseline column, or use the main-table numbers.')

print(f'\n{DATASET}/{BACKBONE}, seed {SEED} — effect of the selection signal\n')
hdr = f'{"method":<7}{"selection":<24}' + ''.join(f'{m:>9}' for m in METRICS)
print(hdr)
print('-' * len(hdr))
for p in PIPELINES:
    if p in baseline:
        b = baseline[p]
        print(f'{p.upper():<7}{"actual request":<24}' +
              ''.join(f'{b[m]:>9.4f}' for m in METRICS))
    h = all_results.get(f'{p}/heldout')
    if h and 'error' not in h:
        print(f'{p.upper():<7}{"held-out simulated":<24}' +
              ''.join(f'{h[m]:>9.4f}' for m in METRICS))
        if p in baseline:
            b = baseline[p]
            print(f'{"":<7}{"delta":<24}' +
                  ''.join(f'{h[m] - b[m]:>+9.4f}' for m in METRICS))
    print('-' * len(hdr))

print('\n% ---- LaTeX (selection-protocol comparison) ----')
print(r'\begin{tabular}{llcccc}')
print(r'\toprule')
print(r'Method & Selection request & Recall@20 & NDCG@20 & MI-BF & MI-NG \\')
print(r'\midrule')
for p in PIPELINES:
    if p in baseline:
        b = baseline[p]
        print(f'{p.upper()} & actual & {b["Recall"]:.4f} & {b["NDCG"]:.4f} & '
              f'{b["MI_BF"]:.4f} & {b["MI_NG"]:.4f} \\\\')
    h = all_results.get(f'{p}/heldout')
    if h and 'error' not in h:
        print(f'{p.upper()} & held-out simulated & {h["Recall"]:.4f} & '
              f'{h["NDCG"]:.4f} & {h["MI_BF"]:.4f} & {h["MI_NG"]:.4f} \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')

---
## Notes for the paper

Suggested wording for Sec. 4.1 / Algorithm 1:

> *Checkpoints are selected by MI-NG on a held-out simulated request:
> a fixed deletion set drawn once with an independent seed, size-matched to
> the actual request, and disjoint from it (injected edges are excluded from
> all simulated draws). The actual request is evaluated once, on the final
> selected checkpoint.*

- If the held-out numbers are close to the actual-selection numbers, report
  the held-out ones as the main results and note the gap — this converts the
  reviewer's "numbers are optimistic" objection into evidence of robustness.
- If they differ materially, the honest move is to feature the held-out
  numbers and discuss the gap; the delta row quantifies exactly how optimistic
  the old protocol was.
- Remaining caveat to state: the selection request is still *simulated
  random* deletions while the actual request is *adversarial* — selection and
  evaluation remain distribution-mismatched, which is the generalization the
  paper claims. No further leakage path remains: training simulations and the
  selection request both exclude the injected edges.